# Study 835 — Spurious Regression 🎭

**Regress one random walk on another — and OLS hands you a "significant" relation that isn't there.**

Granger & Newbold (1974) showed the trap: take two **independent** random walks (each just
a cumulative sum of unrelated coin-flips, so there is *no* relationship between them),
regress one on the other, and the textbook *t*-statistic will call it "significant" the
vast majority of the time — with a high R² to match. It is all an artefact of the two
series **trending** (being non-stationary), not a signal. We simulate 5,000
such pairs and watch the trap spring.

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `73e2821b184c`,
as-of 2026-06-30); the live cells run the fast synthetic controls. Synthetic-only method
demo — no real tape, so it can never earn `REAL` (capped at `NONE` on Signal).*


## 1. Two coin-flip paths that have nothing to do with each other

A *random walk* is just a running total of independent random steps — like a drunkard's path. Build **two** of them from *separate* streams of randomness, so by construction neither knows the other exists. Now regress one on the other. A fair statistical test should call them 'related' only ~5% of the time (the false-positive rate you accept at the 5% level). Watch what OLS actually does.

In [1]:
R = {'lvl_reject': 0.85, 'lvl_reject_x': 17.0, 'lvl_mean_abs_t': 8.99, 'lvl_mean_r2': 0.241, 'dif_reject': 0.053, 'dif_mean_r2': 0.004}
print('LEVELS  (regress y on x):')
print(f"  rejects 'no relation' at |t|>1.96 in {R['lvl_reject']:.0%} of pairs "
      f"(a valid test would: ~5%) -> {R['lvl_reject_x']:.0f}x too often")
print(f"  average |t| = {R['lvl_mean_abs_t']:.1f}   average R2 = {R['lvl_mean_r2']:.2f}")
print('FIRST DIFFERENCES (regress the day-to-day CHANGES) -- the fix:')
print(f"  rejects in {R['dif_reject']:.0%} of pairs (back to ~5%), R2 = {R['dif_mean_r2']:.3f}")

LEVELS  (regress y on x):
  rejects 'no relation' at |t|>1.96 in 85% of pairs (a valid test would: ~5%) -> 17x too often
  average |t| = 9.0   average R2 = 0.24
FIRST DIFFERENCES (regress the day-to-day CHANGES) -- the fix:
  rejects in 5% of pairs (back to ~5%), R2 = 0.004


## 2. See it live — a small simulation, no network

Let's not take the frozen numbers on faith. Simulate a fresh batch of independent random walks right here and run both regressions.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from spurious_regression import data, strategy as st
X, Y = data.independent_walks(1500, n_obs=250, seed=835)
ex = st.regression_experiment(X, Y)
print(f"levels     : rejects in {ex['level']['reject_rate']:.0%} of pairs, "
      f"mean|t| {ex['level']['mean_abs_t']:.1f}, mean R2 {ex['level']['mean_r2']:.2f}")
print(f"differences: rejects in {ex['diff']['reject_rate']:.0%} of pairs, "
      f"mean|t| {ex['diff']['mean_abs_t']:.1f}, mean R2 {ex['diff']['mean_r2']:.3f}")
print('\n-> the level regression is a false-alarm machine; differencing fixes it.')

levels     : rejects in 86% of pairs, mean|t| 9.0, mean R2 0.24
differences: rejects in 7% of pairs, mean|t| 0.8, mean R2 0.004

-> the level regression is a false-alarm machine; differencing fixes it.


## 3. Trending makes it worse — and more data doesn't save you

If the two walks also **drift** (trend) in the same direction, the illusion gets stronger: the level regression rejects **98%** of the time with a mean R² of **0.66**. And — counter to every instinct — *adding data makes it worse*: the spurious *t* grows with √T, so at 1,000 observations the level test rejects **93%** of the time (vs 68% at 50). The differenced test stays at ~5% throughout. A big-*n*, high-*t*, high-R² regression on **levels** is no comfort at all.

In [3]:
R_sweep = [(50, 0.679, 3.99, 0.243, 0.059), (125, 0.787, 6.2, 0.236, 0.052), (250, 0.847, 8.99, 0.241, 0.052), (500, 0.895, 12.81, 0.241, 0.05), (1000, 0.926, 17.99, 0.24, 0.045)]
print('n_obs | level rejects | diff rejects')
for n, lr, mt, r2, dr in R_sweep:
    print(f'{n:>5} | {lr:>11.0%} | {dr:>10.0%}')

n_obs | level rejects | diff rejects
   50 |         68% |         6%
  125 |         79% |         5%
  250 |         85% |         5%
  500 |         90% |         5%
 1000 |         93% |         4%


## 4. The honest verdict

There is **nothing real here** — the series were built independent. The level regression's 'significance' is manufactured by the trending (non-stationary) structure, not by any relationship. The cures are old and simple: **difference to stationarity**, or **test for cointegration** before believing a levels regression. And you certainly can't *trade* the fake relation — the spurious spread is itself a random walk, so a pairs trade on it earns nothing you can distinguish from zero and bleeds costs. **Signal: None** · **Tradability: Mirage** · **Do trending series manufacture false significance? Confirmed.**